In [1]:
# Literature Grounding & Calibration Parameters:
# - Coastal Steel Corrosion Baseline: 0.056 mm/yr (56 um/yr) [Rostron et al.]
# - Temperature Range: Diurnal & seasonal cycling (20°C to 45°C+)
# - Relative Humidity & Moisture Trapping: Sustained kinetics down to 11-23% RH [Rostron et al.]
# - Diurnal Swings: 40% to 100% RH [Saricimen et al.]
# - Seawater & Soil Salinity: 42-45 g/L with high airborne chloride/sulfate contamination (~2000 ppm)
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# Set fixed seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# 1. Load Generated Synthetic Data
df = pd.read_csv('../data/simulated_corrosion.csv')
features = ['temperature_C', 'humidity_pct', 'salinity_gL', 'srb_multiplier']
target = ['corrosion_rate']

# 2. Normalize Features
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(df[features])
y_scaled = scaler_y.fit_transform(df[target])

# 3. Create Rolling Window Sequences (24 hours back to predict next hour)
def create_sequences(X, y, seq_length=24):
    Xs, ys = [], []
    for i in range(len(X) - seq_length):
        Xs.append(X[i:i+seq_length])
        ys.append(y[i+seq_length])
    return np.array(Xs), np.array(ys)

SEQ_LEN = 24
X_seq, y_seq = create_sequences(X_scaled, y_scaled, SEQ_LEN)

# 4. Train/Test Split (80/20)
train_size = int(len(X_seq) * 0.8)
X_train = torch.tensor(X_seq[:train_size], dtype=torch.float32)
X_test  = torch.tensor(X_seq[train_size:], dtype=torch.float32)
y_train = torch.tensor(y_seq[:train_size], dtype=torch.float32)
y_test  = torch.tensor(y_seq[train_size:], dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
print(f"Dataset successfully prepared!\nTrain set: {X_train.shape} | Test set: {X_test.shape}")

Dataset successfully prepared!
Train set: torch.Size([6988, 24, 4]) | Test set: torch.Size([1748, 24, 4])


In [2]:
class PINN_LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2):
        super(PINN_LSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

def physics_informed_loss(y_pred, y_true, x_batch, lambda_arrhenius=0.1, lambda_srb=0.1):
    # 1. Standard Data Loss (MSE)
    mse_loss = nn.MSELoss()(y_pred, y_true)
    
    # 2. Extract inputs from the last timestep of sequence window
    temp_scaled = x_batch[:, -1, 0]
    srb_scaled = x_batch[:, -1, 3]
    
    # 3. Thermal Constraint: Penalize negative predicted corrosion values
    arrhenius_penalty = torch.mean(torch.relu(-y_pred)) 
    
    # 4. Microbial Constraint: Penalize predictions that fail to increase with SRB spikes
    srb_penalty = torch.mean(torch.relu(srb_scaled - y_pred.squeeze()))
    
    return mse_loss + (lambda_arrhenius * arrhenius_penalty) + (lambda_srb * srb_penalty)

model = PINN_LSTM(input_dim=len(features))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Model architecture and custom loss function compiled.")

Model architecture and custom loss function compiled.


In [3]:
epochs = 20
print("Starting Physics-Informed LSTM training...\n")

for epoch in range(epochs):
    model.train()
    total_epoch_loss = 0.0
    for X_b, y_b in train_loader:
        optimizer.zero_grad()
        preds = model(X_b)
        loss = physics_informed_loss(preds, y_b, X_b)
        loss.backward()
        optimizer.step()
        total_epoch_loss += loss.item()
        
    avg_loss = total_epoch_loss / len(train_loader)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{epochs} - Loss: {avg_loss:.4f}")

# Evaluate Test Performance
model.eval()
with torch.no_grad():
    test_preds = model(X_test)
    test_mse = nn.MSELoss()(test_preds, y_test).item()
    print(f"\nTraining Complete! Test MSE Loss: {test_mse:.4f}")

# Save Model Weights
torch.save(model.state_dict(), 'pinn_lstm_model.pth')
print("Model weights successfully saved to 'model/pinn_lstm_model.pth'!")

Starting Physics-Informed LSTM training...

Epoch 01/20 - Loss: 0.5452
Epoch 05/20 - Loss: 0.3768
Epoch 10/20 - Loss: 0.3716
Epoch 15/20 - Loss: 0.3821
Epoch 20/20 - Loss: 0.3735

Training Complete! Test MSE Loss: 0.1763
Model weights successfully saved to 'model/pinn_lstm_model.pth'!
